Context:

Environment: Databricks (PySpark), Unity Catalog enabled.
Source: <CATALOG>.<SCHEMA>.<table> (fallback to in-memory sample if missing).
Destination/UC namespace: <CATALOG>.<SCHEMA> (I will provide values).
Requirements:
Create a modular ETL with functions: extract_data, transform_data, apply_quality_checks_and_log, load_data, and a run_pipeline orchestrator.

Transform rules:
Filter out rows with null (I will provide columns).
Detect data types in the columns of the table.
Filter out duplicates based on all columns.

Data-quality:
Accept a list of rules that are either (column, op, threshold) or free-form SQL expr.
For any violation, append one row per violating record to a Delta log table with columns:
job_run_id, rule_name, rule_severity, rule_type, column_name, threshold, actual_value, key_json, source_table, target_table, stage, event_ts.
If a rule’s severity is ERROR, also append the full rows to a Delta quarantine table and exclude them from the final load.
Perform incremental load update and insert for saving log and quarantine data to delta tables.
Carry a job_run_id = uuid4().

Targets:
Clean output table: <CATALOG>.<SCHEMA>.<TABLE> (overwrite each run).
Log table: <CATALOG>.<SCHEMA>.etl_exception_log (append).
Quarantine table: <CATALOG>.<SCHEMA>.default_quarantine (append).
Create tables if they don’t exist.
Parameters (easy to change at top of script): <CATALOG>, <SCHEMA>, SOURCE_TABLE, TARGET_TABLE, LOG_TABLE, QUARANTINE_TABLE, KEY_COLS, and DQ_RULES.
Add basic error handling to each stage; fail the job with a clear message on unrecoverable errors.
Include short comments explaining each section and key decisions.

load_data rules:
Perform incremental load as update and insert (UPSERT) from source to target.

Execute pipeline as:
if __name__ == "__main__":
    run_pipeline()

Acceptance criteria:
Running run_pipeline() performs: Extract → Transform → Validate/Log/Quarantine → Load.
After a run, I can:
Query <CATALOG>.<SCHEMA>.<TABLE> to see only clean rows.
Query <CATALOG>.<SCHEMA>.etl_exception_log to view detailed violations.
Query <CATALOG>.<SCHEMA>.default_quarantine to inspect error rows.
I can add/edit rules in DQ_RULES and re-run without code changes elsewhere.